## MLP - 多层感知机


In [9]:
import torch
import torch.nn as nn # 神经网络模块, 提供各种神经网络层的实现
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# =====================
# 1. 定义 MLP 模型
# =====================
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_hidden_layers=2, dropout=0.2):
        """
        多层感知机模型
        :param input_dim: 输入特征维度
        :param hidden_dim: 隐藏层维度
        :param output_dim: 输出维度（分类数）
        :param num_hidden_layers: 隐藏层数量
        :param dropout: Dropout 比率
        """
        super(MLP, self).__init__()
        
        layers = []
        # 输入层 -> 第一个隐藏层
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        
        # 中间隐藏层
        for _ in range(num_hidden_layers - 1): # _ 符号表示循环变量不需要使用，从0开始循环到 num_hidden_layers - 1，如果 num_hidden_layers=2，则只会执行一次循环，添加一个隐藏层
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        
        # 输出层
        layers.append(nn.Linear(hidden_dim, output_dim))
        
        self.network = nn.Sequential(*layers) # nn.Sequential 是一个容器模块，它将多个层按顺序组合在一起，形成一个新的模块。通过 *layers 将列表中的层展开传递给 nn.Sequential，使得这些层按照定义的顺序依次执行，从而构建了整个 MLP 网络结构。
    
    def forward(self, x): # forward 方法定义了数据如何通过网络进行前向传播，x 是输入数据，返回值是模型的输出
        return self.network(x)


# =====================
# 2. 超参数配置
# =====================
config = {
    'input_dim': 20,          # 输入特征维度
    'hidden_dim': 64,         # 隐藏层神经元数量
    'output_dim': 3,          # 输出类别数
    'num_hidden_layers': 2,   # 隐藏层数量
    'dropout': 0.2,           # Dropout 比率, Dropout用于防止过拟合，通过在训练过程中随机丢弃一部分神经元来提高模型的泛化能力
    'learning_rate': 1e-3,    # 学习率
    'batch_size': 32,         # 批次大小
    'epochs': 50,             # 训练轮数
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}


# =====================
# 3. 创建模型、损失函数和优化器
# =====================
model = MLP(
    input_dim=config['input_dim'],
    hidden_dim=config['hidden_dim'],
    output_dim=config['output_dim'],
    num_hidden_layers=config['num_hidden_layers'],
    dropout=config['dropout']
).to(config['device'])


criterion = nn.CrossEntropyLoss()  # 分类任务损失函数，损失函数用于衡量模型预测结果与真实标签之间的差异，CrossEntropyLoss适用于多分类问题，它结合了LogSoftmax和NLLLoss，能够有效地计算预测概率分布与真实标签分布之间的交叉熵损失。
optimizer = optim.Adam(model.parameters(), lr=config['learning_rate']) # Adam优化器是一种自适应学习率优化算法，它结合了动量和RMSProp的优点，能够在训练过程中自动调整每个参数的学习率，从而加快收敛速度并提高模型性能。
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)  # 学习率调度器, 每训练20个epoch将学习率减半


# =====================
# 4. 生成模拟数据（实际使用时替换为真实数据）
# =====================
num_samples = 1000
X = torch.randn(num_samples, config['input_dim'])
y = torch.randint(0, config['output_dim'], (num_samples,))

dataset = TensorDataset(X, y)
train_size = int(0.8 * num_samples)
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, num_samples - train_size])

train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)


# =====================
# 5. 训练循环
# =====================
for epoch in range(config['epochs']):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(config['device']), batch_y.to(config['device'])
        
        # 前向传播
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y) # 计算损失
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward() # 计算梯度
        optimizer.step() # 更新参数
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
    
    scheduler.step() # 更新学习率
    
    # =====================
    # 6. 验证
    # =====================
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(config['device']), batch_y.to(config['device'])
            outputs = model(batch_X)
            _, predicted = outputs.max(1)
            val_total += batch_y.size(0)
            val_correct += predicted.eq(batch_y).sum().item()
    
    print(f"Epoch [{epoch+1}/{config['epochs']}] | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Train Acc: {100.*correct/total:.2f}% | "
          f"Val Acc: {100.*val_correct/val_total:.2f}%")

          




Epoch [1/50] | Train Loss: 1.1240 | Train Acc: 33.12% | Val Acc: 33.50%
Epoch [2/50] | Train Loss: 1.0991 | Train Acc: 37.50% | Val Acc: 33.50%
Epoch [3/50] | Train Loss: 1.0884 | Train Acc: 37.88% | Val Acc: 35.50%
Epoch [4/50] | Train Loss: 1.0752 | Train Acc: 42.00% | Val Acc: 34.00%
Epoch [5/50] | Train Loss: 1.0745 | Train Acc: 41.50% | Val Acc: 33.50%
Epoch [6/50] | Train Loss: 1.0636 | Train Acc: 44.88% | Val Acc: 34.00%
Epoch [7/50] | Train Loss: 1.0561 | Train Acc: 44.12% | Val Acc: 34.00%
Epoch [8/50] | Train Loss: 1.0438 | Train Acc: 46.88% | Val Acc: 33.00%
Epoch [9/50] | Train Loss: 1.0423 | Train Acc: 46.62% | Val Acc: 34.50%
Epoch [10/50] | Train Loss: 1.0381 | Train Acc: 45.50% | Val Acc: 34.00%
Epoch [11/50] | Train Loss: 1.0301 | Train Acc: 48.50% | Val Acc: 34.50%
Epoch [12/50] | Train Loss: 1.0261 | Train Acc: 49.88% | Val Acc: 33.00%
Epoch [13/50] | Train Loss: 1.0162 | Train Acc: 50.25% | Val Acc: 35.00%
Epoch [14/50] | Train Loss: 1.0038 | Train Acc: 52.62% | Val

In [6]:
# 加入残差连接的 MLP 模型
class MLPResNet(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        num_hidden_layers=2,
        dropout=0.2
    ):
        super(MLP, self).__init__()

        # 第一层：20 -> 64
        self.input_layer = nn.Linear(input_dim, hidden_dim)

        # 中间隐藏层：64 -> 64
        self.hidden_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim)
            for _ in range(num_hidden_layers - 1) 
        ])

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        # 输出层：64 -> 3
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):

        # ========================
        # 第一隐藏层
        # ========================
        x = self.input_layer(x)
        x = self.relu(x)
        x = self.dropout(x)

        # 此时 x.shape = [batch_size, 64]

        # ========================
        # 残差隐藏层
        # ========================
        for layer in self.hidden_layers:

            # 保存残差
            residual = x

            # F(x)
            x = layer(x)
            x = self.relu(x)
            x = self.dropout(x)

            # 残差连接
            x = x + residual

        # ========================
        # 输出层
        # ========================
        x = self.output_layer(x)

        return x

# PyTorch 常用损失函数

损失函数用于衡量模型预测值与真实值之间的差异。不同任务应选择合适的损失函数。

## 1. 均方误差损失：`nn.MSELoss`

均方误差（MSE）常用于回归任务：

$$
L = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2
$$

```python
criterion = nn.MSELoss()
loss = criterion(prediction, target)
```

其中，$\hat{y}$ 是模型预测值，$y$ 是真实值。该损失会放大较大的误差，因此对异常值比较敏感。

---

## 2. 平均绝对误差损失：`nn.L1Loss`

平均绝对误差（MAE）同样常用于回归任务：

$$
L = \frac{1}{N}\sum_{i=1}^{N}|y_i - \hat{y}_i|
$$

```python
criterion = nn.L1Loss()
loss = criterion(prediction, target)
```

相比 MSE，L1 损失对异常值相对不敏感。

---

## 3. 平滑 L1 损失：`nn.SmoothL1Loss`

平滑 L1 损失结合了 MSE 和 L1 损失的优点，常用于目标检测和回归任务：

$$
L(x, y) =
\begin{cases}
\frac{1}{2}\frac{(x-y)^2}{\beta}, & |x-y| < \beta \\
|x-y| - \frac{1}{2}\beta, & \text{其他情况}
\end{cases}
$$

```python
criterion = nn.SmoothL1Loss()
loss = criterion(prediction, target)
```

---

## 4. 交叉熵损失：`nn.CrossEntropyLoss`

交叉熵损失常用于多分类任务。对于真实类别 $y$，其形式为：


$$ H(p,q)=-\sum_{i=1}^{C}p_i\log q_i $$

$$
L = -\log\frac{e^{z_y}}{\sum_{j=1}^{C}e^{z_j}}
$$

其中，$z_y$ 是真实类别对应的模型输出值，$C$ 是类别总数。

```python
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)
```

使用时需要注意：

- 模型输出应为未经 Softmax 的 logits；
- 标签应为类别索引，例如 `0`、`1`、`2`；
- 输出形状通常为 `(batch_size, num_classes)`；
- 标签形状通常为 `(batch_size,)`。

当前 MLP 模型是三分类任务，因此使用 `nn.CrossEntropyLoss()` 是合适的。

---

## 5. 二元交叉熵损失：`nn.BCELoss`

`BCELoss` 用于二分类任务，要求模型输出已经经过 Sigmoid，且取值范围为 $[0, 1]$：

$$
L = -\left[y\log(\hat{y}) + (1-y)\log(1-\hat{y})\right]
$$

```python
criterion = nn.BCELoss()
loss = criterion(probability, target.float())
```

---

## 6. 带 Sigmoid 的二元交叉熵：`nn.BCEWithLogitsLoss`

`BCEWithLogitsLoss` 将 Sigmoid 和二元交叉熵合并，数值稳定性更好，通常更推荐：

```python
criterion = nn.BCEWithLogitsLoss()
loss = criterion(logits, target.float())
```

使用该损失时，模型最后一层不要再手动添加 `Sigmoid`。

---

## 7. 负对数似然损失：`nn.NLLLoss`

`NLLLoss` 通常与 `LogSoftmax` 配合使用：

负对数似然损失（NLLLoss）主要用于分类任务，尤其是模型已经输出对数概率的场景。
```python
criterion = nn.NLLLoss()
output = torch.log_softmax(logits, dim=1)
loss = criterion(output, target)
```

其定义为：

$$
L = -\log P(y \mid x)
$$

---

## 8. KL 散度损失：`nn.KLDivLoss`

KL 散度用于衡量两个概率分布之间的差异，常用于知识蒸馏：

$$
D_{KL}(P\|Q) = \sum_i P_i\log\frac{P_i}{Q_i}
$$

```python
criterion = nn.KLDivLoss()
input_log_probability = torch.log_softmax(student_logits, dim=1)
target_probability = torch.softmax(teacher_logits, dim=1)
loss = criterion(input_log_probability, target_probability)
```

模型输入通常需要使用 `log_softmax`，目标需要是概率分布。

---

## 9. 余弦嵌入损失：`nn.CosineEmbeddingLoss`

余弦嵌入损失用于判断两个向量是否相似，常用于相似度学习、文本匹配和人脸识别：

```python
criterion = nn.CosineEmbeddingLoss()
loss = criterion(input_1, input_2, label)
```

其中，`label` 通常取 `1` 或 `-1`：

- `1` 表示两个向量应该相似；
- `-1` 表示两个向量应该不相似。


# 模型评估与验证


# 模型验证与评估：从基础到进阶

模型验证评估是训练过程中最关键的"反馈环节"——它告诉你模型到底学到了什么、哪里出了问题、该往哪个方向优化。下面从底层概念到实战策略，逐层展开。

---

## 基础：混淆矩阵——所有指标的"地基"

在理解任何指标之前，必须先掌握混淆矩阵。它是一张 2×2 的表格，将模型的预测结果与真实标签交叉对比，产生四种基本结果：

| | 预测为正类 | 预测为负类 |
|:---:|:---:|:---:|
| **真实为正类** | TP（真正例）：猜对了 | FN（假负例）：漏掉了 |
| **真实为负类** | FP（假正例）：误报了 | TN（真负例）：排除对了 |

> 💡 **记忆技巧**：T/F 表示"猜对/猜错"，P/N 表示"猜的是正类/负类"。比如 FP = False Positive = 猜错了，猜的是正类 = 误报

所有后续指标（准确率、精确率、召回率等）都是从这四个值计算出来的。混淆矩阵的价值在于让你清晰看到模型错在哪——是误报太多（FP 多），还是漏报太多（FN 多）。

---

## 核心指标详解

### 准确率（Accuracy）

**公式：** Accuracy = (TP + TN) / 总样本数

**含义：** 所有样本中，模型预测正确的比例。就像考试的"总分"——不管哪一科，答对就算分。

**适用场景：**

- 类别分布均衡的分类任务（如手写数字识别、MNIST）
- 模型初步评估时的快速概览

**致命缺陷：** 样本不均衡时会"骗人"。比如欺诈检测中 99% 是正常交易，模型全部预测"正常"也能拿到 99% 准确率，但实际上一个欺诈都没抓到。

---

### 精确率（Precision）

**公式：** Precision = TP / (TP + FP)

**含义：** 模型预测为正类的样本中，真正是正类的比例。通俗说就是——模型说"是"的时候，到底准不准。

**适用场景（误报代价高的场景）：**

- 垃圾邮件过滤：不想把正常邮件误判为垃圾邮件
- 内容审核/自动封禁：误封正常用户会引发投诉
- 广告投放：不想把预算浪费在非目标用户上

---

### 召回率（Recall）

**公式：** Recall = TP / (TP + FN)

**含义：** 所有真实的正类样本中，被模型成功找出来的比例。通俗说就是——该找的目标，找全了没有。

**适用场景（漏报代价高的场景）：**

- 癌症筛查：漏掉一个患者可能危及生命
- 欺诈检测：漏掉一笔欺诈交易就是直接损失
- 安防告警：漏掉风险事件后果严重

---

### 精确率 vs 召回率：跷跷板关系

这两个指标是一对"跷跷板"——提高一个往往会降低另一个：

- 放宽阈值（更容易判为正类）→ 召回率↑（漏得少），但精确率↓（误报多）
- 收紧阈值（更严格才判为正类）→ 精确率↑（误报少），但召回率↓（漏得多）

**选择哪个优先，取决于业务代价：误报贵还是漏报贵？**

---

### F1 分数（F1-Score）

**公式：** F1 = 2 × (Precision × Recall) / (Precision + Recall)

**含义：** 精确率和召回率的调和平均数，综合衡量模型在"准"和"全"之间的平衡能力。

**为什么用调和平均而不是普通平均？** 因为调和平均对"短板"更敏感——如果精确率 0.9 但召回率只有 0.3，F1 会被拉得很低，真实反映模型的失衡状态。

**适用场景：**

- 误报和漏报代价都不能忽视的场景
- 样本不均衡时的综合评估
- 信息检索、文本分类、命名实体识别等 NLP 任务

> 💡 **扩展**：Fβ 分数允许你给精确率和召回率赋予不同权重。F2 更侧重召回率，F0.5 更侧重精确率。

---

### AUC（Area Under ROC Curve）

**ROC 曲线：** 以假阳性率（FPR = FP/(FP+TN)）为横轴，真阳性率（TPR = Recall）为纵轴，描绘模型在所有可能阈值下的表现。

**AUC 值：** ROC 曲线下的面积，范围 0.5~1：

- AUC = 1：完美分类器
- AUC = 0.5：等同于随机猜测
- AUC < 0.5：比随机还差（但反转预测即可改善）

**核心优势：**

- **阈值无关：** 不依赖某个固定阈值，综合评估模型在所有阈值下的表现
- **对类别不平衡鲁棒：** 即使正负样本比例悬殊，AUC 仍能提供公正评估
- **便于模型对比：** 直接比较 AUC 大小即可判断模型优劣
- **直观理解：** 随机抽取一个正样本和一个负样本，模型给正样本打更高分的概率就是 AUC

**局限性：**

- 作为全局指标，无法反映特定阈值下的表现
- 在极度不平衡场景下（如罕见病检测，正样本 <1%），PR 曲线和 AP 值（平均精确率）可能更有参考价值

---

## 指标选择速查表

| 场景 | 优先指标 | 原因 |
|:---|:---:|:---|
| 癌症筛查、欺诈检测 | 召回率 | 漏报代价极高 |
| 垃圾邮件、内容审核 | 精确率 | 误报代价极高 |
| 综合评估、模型选型 | F1 分数 | 需要平衡两者 |
| 多模型对比 | AUC | 阈值无关，便于排序 |
| 类别均衡的简单分类 | 准确率 | 简单直观 |

---

## 监控过拟合与欠拟合

通过对比训练集和验证集的表现，可以快速诊断模型状态：

### 三种典型状态

| 状态 | 训练集表现 | 验证集表现 | 诊断 |
|:---|:---:|:---:|:---|
| 欠拟合 | 差 | 差 | 模型太简单，连训练数据都没学会 |
| 正确拟合 | 好 | 好（差距小） | 模型学到了有效规律 |
| 过拟合 | 极好（如 99%） | 差（如 70%） | 模型死记硬背了训练数据的噪声 |

通常训练集和验证集的准确率差距超过 5% 就可以作为过拟合的警告信号，但具体阈值需根据任务调整。

### 可视化诊断方法

最常用的是绘制训练/验证 Loss 和 Accuracy 曲线：

- **正常：** 两条曲线同步下降，差距较小
- **过拟合：** 训练 Loss 持续下降，验证 Loss 在某轮次后开始回升
- **欠拟合：** 两条曲线都很高，且下降缓慢

### 应对策略

**过拟合的解决方案：**

- **数据层面：** 增加训练数据、数据增强（随机翻转、裁剪等）
- **模型层面：** 增加 Dropout、L1/L2 正则化、减小模型复杂度
- **训练层面：** 早停法（Early Stopping）——验证集指标连续 N 个 epoch 不再提升时停止训练

**欠拟合的解决方案：**

- 增加特征复杂度（如构建交叉特征）
- 换更复杂的模型
- 减小正则化强度
- 增加训练轮数

---

## 值得扩展的概念

### 多分类场景的指标计算

当类别超过两个时（如你的代码中 output_dim=3），每个指标有三种平均方式：

- **宏平均（Macro）：** 对每个类别单独计算指标，再取简单平均——对每个类别一视同仁
- **微平均（Micro）：** 将所有类别的 TP/FP/FN 累加后统一计算——更关注样本量大的类别
- **加权平均（Weighted）：** 按每个类别的样本数加权平均

### 回归任务的评估指标

如果你的任务是回归（预测连续值）而非分类，常用指标包括：

- **MAE（平均绝对误差）：** 直观，对所有误差一视同仁
- **MSE（均方误差）：** 对大误差惩罚更重
- **RMSE（均方根误差）：** 与目标变量同量纲，便于解释
- **R²（决定系数）：** 模型解释了多少方差

### 交叉验证

单次 train/val 划分可能受随机性影响。K-Fold 交叉验证将数据分成 K 份，轮流用 K-1 份训练、1 份验证，取 K 次结果的平均值，使评估更稳健。对于不均衡数据，应使用分层 K-Fold，确保每份数据的类别比例一致。

### LogLoss（对数损失）

前面的指标都基于"分类结果"（0 或 1），但很多场景需要模型输出概率（如"违约概率 30%"）。LogLoss 衡量的就是模型预测概率的精准度——越接近真实标签的概率越好，对"过度自信的错误预测"惩罚极重。